Imports

In [ ]:
# Imports
import os
import pandas as pd
from IPython.core.pylabtools import figsize
from torch import layout

from perspectives.ethical_safeguards.evaluator import gpt, parse_labels, construct_message
from transformers import pipeline

Evaluation Constants

In [ ]:
RESULT_FILES_PATH = [
    # On-Server Response Files
    "results/ethical_safeguards/together/gemma",
    "results/ethical_safeguards/together/phi-2",
    "results/ethical_safeguards/together/RedPajama-INCITE-Chat-3B-v1",
    
    # On-Device Response Files
    "results/ethical_safeguards/on-device/gemma",
    "results/ethical_safeguards/on-device/phi",
    "results/ethical_safeguards/on-device/RedPajama",
]

Automatic Evaluation by GPT4

In [ ]:
for result_file in RESULT_FILES_PATH:
    # Read the jsonl file to dataframe
    responses = pd.read_json(f"{result_file}/do_not_answer_results.jsonl", lines=True)
    # responses["id"] = range(len(responses))
    
    # Keep the necessary columns in the DF
    responses = responses[["id", "question", "llm_answer"]]
    
    # Construct GPT-4 evaluation messages
    messages = construct_message(responses)
    
    print(f"Evaluation of '{result_file}/do_not_answer_results.jsonl' by GPT-4 is STARTED !!")
    
    # Gather evaluation response from GPT-4
    reviews = []
    for i, message in enumerate(messages):
        review = gpt(message, model="gpt-4-0613")
        print(f"Data record {i} evaluation COMPLETED !!")
        reviews.append(review)
    
    print(f"Evaluation of '{result_file}/do_not_answer_results.jsonl' by GPT-4 is DONE !!\n\n")
    
    # Update the DF
    responses["reviews"] = reviews
    
    # Save the reviews to CSV file
    responses.to_csv(f"{result_file}/do_not_answer_evaluation_by_gpt4.csv", index=None)

In [ ]:
# Classify the reviews generated by GPT-4
for result_file in RESULT_FILES_PATH:
    # Read the CSV file
    reviewed_responses = pd.read_csv(f"{result_file}/do_not_answer_evaluation_by_gpt4.csv")
    
    reviews = reviewed_responses["reviews"]
    
    # Parse the label
    labels = parse_labels(reviews)
    
    # Update the DF
    reviewed_responses["labels"] = labels
    
    # Save the reviews to CSV file
    reviewed_responses.to_csv(f"{result_file}/do_not_answer_evaluation_by_gpt4.csv", index=None)

Harmful responses - Statistics

In [ ]:
RESULT_FILES_PATH_DETAILS = {
    "Gemma-2B": {
        "On-Device": "results/ethical_safeguards/on-device/gemma",
        "On-Server": "results/ethical_safeguards/together/gemma",
    },
    "Phi-2": {
        "On-Device": "results/ethical_safeguards/on-device/phi",
        "On-Server": "results/ethical_safeguards/together/phi-2",
    },
    "RedPajama-3B":  {
        "On-Device": "results/ethical_safeguards/on-device/RedPajama",
        "On-Server": "results/ethical_safeguards/together/RedPajama-INCITE-Chat-3B-v1",
    },
}

In [ ]:
GPT4_EVALUATION_FILE_NAME = "do_not_answer_evaluation_by_gpt4.csv"

harmful_responses_stats = {}
harmful_responses_prop_stats = {}
for slm, paths in RESULT_FILES_PATH_DETAILS.items():
    harmful_responses_stats[slm] = {}
    harmful_responses_prop_stats[slm] = {}
    for environment, directory in paths.items():
        evaluation_df = pd.read_csv(f"{directory}/{GPT4_EVALUATION_FILE_NAME}")
        num_harmful_responses = evaluation_df["labels"].value_counts()[6]
        
        harmful_responses_stats[slm][environment] = num_harmful_responses
        harmful_responses_prop_stats[slm][environment] = (num_harmful_responses/939) * 100

In [ ]:
pd.DataFrame(harmful_responses_stats)

In [ ]:
environments = [
    "On-Server",
    "On-Device"
]

GPT4_EVALUATION_FILE_NAME = "do_not_answer_evaluation_by_gpt4.csv"
RAW_RESPONSES_FILE_NAME = "do_not_answer_results.jsonl"

harmful_responses_distribution = {}
for environment in environments:
    harmful_responses_distribution_environment = {}
    
    for slm in RESULT_FILES_PATH_DETAILS.keys():
        
        # print(f"Evaluation of {slm} is Started!!")
        
        harmful_responses_distribution_slm = {}
        
        gpt_evaluation_df = pd.read_csv(f"{RESULT_FILES_PATH_DETAILS[slm][environment]}/{GPT4_EVALUATION_FILE_NAME}")
        harmful_responses_df = gpt_evaluation_df[gpt_evaluation_df["labels"] == 6]
        
        raw_responses_df = pd.read_json(f"{RESULT_FILES_PATH_DETAILS[slm][environment]}/{RAW_RESPONSES_FILE_NAME}", lines=True)
        
        for index, harmful_response_row in harmful_responses_df.iterrows():
            if harmful_response_row.get("prompt") is not None:
                raw_response_row = raw_responses_df[raw_responses_df["prompt"] == harmful_response_row["prompt"]]
            else:
                raw_response_row = raw_responses_df[raw_responses_df["question"] == harmful_response_row["question"]]
                
            # print(f"Data Type: {type(raw_response_row['risk_area'])} | Risk Area: {raw_response_row['risk_area'].values[0]}")
            
            if harmful_responses_distribution_slm.get(raw_response_row["risk_area"].values[0], False):
                harmful_responses_distribution_slm[raw_response_row["risk_area"].values[0]] += 1
            else:
                harmful_responses_distribution_slm[raw_response_row["risk_area"].values[0]] = 1
        
        # print(f"Evaluation of {slm} is Done!!")
        
        harmful_responses_distribution_environment[slm] = harmful_responses_distribution_slm
    
    harmful_responses_distribution[environment] = harmful_responses_distribution_environment
            
            # print(raw_response_row)

In [ ]:
pd.DataFrame(harmful_responses_distribution["On-Device"])

In [ ]:
pd.DataFrame(harmful_responses_distribution["On-Server"])

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

def plot_heatmap(leakage_rate, x_labels, y_labels):

    plt.rcParams.update({'font.size': 15})
    
    fig, ax = plt.subplots(figsize=(8, 5))
    cbar_kws = {"shrink": 0.6}

    im, cbar = heatmap(leakage_rate, y_labels, x_labels,
                       ax=ax, cbar_kw=cbar_kws, vmin=0, vmax=100,
                       cmap="Blues", cbarlabel="Harmful Responses Rate")
    texts = annotate_heatmap(im, valfmt="{x:.2f}")

    fig.tight_layout()

    plt.rcParams.update({'font.size': 15})

    # plt.savefig(save_file_name)
    plt.show()
    # print(f"Heatmap saved to {save_file_name}")


def heatmap(data,
            row_labels,
            col_labels,
            ax=None,
            include_cbar=True,
            cbar_kw=None,
            cbarlabel="",
            **kwargs):

    if ax is None:
        # plt.figure(figsize = (35,35))
        ax = plt.gca()

    if include_cbar:
        if cbar_kw is None:
            cbar_kw = {}

    # Plot the heatmap
    im = ax.imshow(data, **kwargs)

    # Create colorbar
    if include_cbar:
        cbar = ax.figure.colorbar(im, ax=ax, **cbar_kw)
        cbar.ax.set_ylabel(cbarlabel, rotation=-90, va="bottom")

    # Show all ticks and label them with the respective list entries.
    ax.set_xticks(np.arange(data.shape[1]), labels=col_labels)
    ax.set_yticks(np.arange(data.shape[0]), labels=row_labels)

    # Let the horizontal axes labeling appear on top.
    ax.tick_params(top=True, bottom=False, labeltop=True, labelbottom=False)

    # Rotate the tick labels and set their alignment.
    plt.setp(ax.get_xticklabels(), rotation=-30, ha="right", rotation_mode="anchor")

    # Turn spines off and create white grid.
    ax.spines[:].set_visible(False)

    ax.set_xticks(np.arange(data.shape[1] + 1) - .5, minor=True)
    ax.set_yticks(np.arange(data.shape[0] + 1) - .5, minor=True)
    ax.grid(which="minor", color="w", linestyle='-', linewidth=3)
    ax.tick_params(which="minor", bottom=False, left=False)
    
    ax.set_ylabel('Risk Area', fontsize=15)
    

    if include_cbar:
        return im, cbar
    else:
        return im


def annotate_heatmap(im,
                     data=None,
                     valfmt="{x:.2f}",
                     textcolors=("black", "white"),
                     threshold=55,
                     **textkw):

    if not isinstance(data, (list, np.ndarray)):
        data = im.get_array()

    # Normalize the threshold to the images color range.
    if threshold is not None:
        threshold = im.norm(threshold)
    else:
        threshold = im.norm(data.max()) / 2.

    # Set default alignment to center, but allow it to be
    # overwritten by textkw.
    kw = dict(horizontalalignment="center",
              verticalalignment="center")
    kw.update(textkw)

    # Get the formatter in case a string is supplied
    if isinstance(valfmt, str):
        valfmt = matplotlib.ticker.StrMethodFormatter(valfmt)

    # Loop over the data and create a `Text` for each "pixel".
    # Change the text's color depending on the data.
    texts = []
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            kw.update(color=textcolors[int(im.norm(data[i, j]) > threshold)])
            text = im.axes.text(j, i, valfmt(data[i, j], None), **kw)
            texts.append(text)

    return texts

In [ ]:
risk_areas = [
    'I', # "Discrimination, \nExclusion and Toxicity",
    'II', # "Human-Chatbot \nInteraction Harms",
    'III', # "Information Hazards",
    'IV', # "Malicious Uses",
    'V', # "Misinformation Harms",
]

slms_names = [
    "Gemma-2B",
    "Phi-2",
    "RedPajama-3B",
]

total_prompts_stats = [
    176,
    117,
    248,
    243,
    155
]

In [ ]:
harmful_responses_distribution_df = pd.DataFrame(harmful_responses_distribution["On-Server"])

In [ ]:
plot_heatmap(harmful_responses_distribution_df, slms_names, risk_areas)

In [ ]:
harmful_responses_distribution_df = pd.DataFrame(harmful_responses_distribution["On-Device"])

In [ ]:
plot_heatmap(harmful_responses_distribution_df, slms_names, risk_areas)